# Step 1 — Disfluency BIO labeling (IP/FS/RP/RM/TH/CF/UC/PL/SP taxonomy)

`data/normalized/intent_dataset_normalized.jsonl` already carries a gold `disfluencies` field per record — each entry is `{"tag": <IP|FS|RP|RM|TH|CF|UC|PL|SP>, "token"|"tokens": <span text>}` describing exactly which surface span carries which disfluency type. This notebook converts that gold annotation directly into token-level BIO tags over `text` (a prior version of this notebook re-derived labels via regex heuristics, which silently missed most `RP`/`RM` repairs and any non-`eh`/`anu` bridge filler — using the gold field instead is both more accurate and simpler).

Taxonomy (expanded from the original 5-tag IP/FS/RC/RP/RM scheme to cover all 12 categories in `data/raw/disfluencies.json` — the old narrow `VALID_LABELS` was silently coercing `fillers`/`thinking`/`confirmation`/`politeness`/`change_of_mind` spans to `O`, which is why common fillers like `eh`/`anu` were missing from trained output):
- `IP` (interregnum/filler) — discourse fillers/hesitation, single or multi-word (`eh`, `anu`, `um`, `hmm`, `bentar`, `sebentar ya...`)
- `FS` (false start) — hyphen-truncated partial word immediately followed by its completed form (e.g. `mi- mie`)
- `RP`/`RM` (reparandum/repair) — `repetition`, `self_correction`, `change_of_mind`, `repair` all collapse into this pair. Repetition splits its matched span into `RP` (first occurrence) + `RM` (second); the other three tag their whole phrase `RM` (they're single correction-signal phrases, not linked wrong/right spans)
- `TH` (thinking) — stalling phrases while deciding (`sebentar ya...`, `saya pilih dulu...`)
- `CF` (confirmation) — phrases bridging to a confirmed choice (`jadi...`, `oke berarti...`)
- `UC` (uncertainty) — hedging words (`kayaknya`, `mungkin`, `sepertinya`)
- `PL` (politeness) — politeness markers (`tolong`, `boleh`, `permisi`)
- `SP` (spoken particle) — discourse particles (`dong`, `deh`, `ya`, `kok`, `sih`)

Encoder target for downstream training: `indobenchmark/indobert-base-p1` (matches the NER backbone, reverted in `9f1dd1a`).

In [1]:
import json
from collections import Counter
from pathlib import Path

SRC = Path("../data/normalized/augmented_ordering_dataset_normalized.jsonl")
OUT_DIR = Path("../data/disfluency")

## Step 2 — Span matching helpers

Each gold `disfluencies` entry holds the disfluency span as raw text (`token` for a single/multi-word span; `tokens` is also supported for any future entry needing multiple non-contiguous words, though the current generator only ever emits `token`). `strip_punct` normalizes both the sentence tokens and the gold span text the same way before matching, so trailing commas on tokens like `"eh,"` don't break equality.

In [2]:
def tokenize(text: str) -> list[str]:
    return text.split()


def strip_punct(tok: str) -> str:
    return tok.strip(",.;:!?").lower()


def get_span_tokens(d: dict) -> list[str]:
    if "tokens" in d:
        return [strip_punct(t) for t in d["tokens"]]
    return [strip_punct(t) for t in d["token"].split()]

## Step 3 — BIO labeler from gold spans

`match_spans` locates each gold disfluency's token span inside the sentence and marks those positions used, so a repeated surface form (e.g. `eh` appearing twice, once inside `eh ya` and once standalone) binds to the correct occurrence instead of always the first. Gold entries aren't guaranteed to be in left-to-right text order (verified: 1/5100 records has `RP`/`RM`/`IP` listed out of text order), so matching is order-independent over the disfluencies list rather than a single sequential scan.

In [3]:
def match_spans(toks: list[str], disfluencies: list[dict]) -> list[tuple[int, int, str]] | None:
    """Match each gold disfluency entry to a token span, returning (start, end, tag)
    triples (end exclusive). Returns None if any entry can't be matched."""
    used = [False] * len(toks)
    spans = []
    for d in disfluencies:
        span = get_span_tokens(d)
        L = len(span)
        found = -1
        for i in range(len(toks) - L + 1):
            if any(used[i : i + L]):
                continue
            if toks[i : i + L] == span:
                found = i
                break
        if found == -1:
            return None
        for k in range(found, found + L):
            used[k] = True
        spans.append((found, found + L, d["tag"]))
    return spans


def full_label(raw_text: str, disfluencies: list[dict]) -> tuple[list[str], list[str]]:
    raw_toks = tokenize(raw_text)
    toks = [strip_punct(t) for t in raw_toks]
    labels = ["O"] * len(toks)
    spans = match_spans(toks, disfluencies)
    if spans is None:
        return raw_toks, None
    for start, end, tag in spans:
        labels[start] = f"B-{tag}"
        for k in range(start + 1, end):
            labels[k] = f"I-{tag}"
    return raw_toks, labels

## Step 4 — Validation

Auto-corrects orphan `I-` tags (continuation tag with no preceding `B-`/`I-` of the same type) to `B-`, and any unrecognized label value to `O`. `match_spans`/`full_label` never produce orphans by construction (each matched span always starts with `B-`), but this guards against future changes silently breaking BIO well-formedness. `VALID_LABELS` must cover every tag `generate_ordering_dataset.py`'s `DISF_TAG_MAP` can emit — a narrower set here silently drops real disfluencies to `O` (this happened previously: `fillers`/`thinking`/`confirmation`/`politeness`/`change_of_mind` spans were coerced away because the old 5-tag `VALID_LABELS` didn't include them).

In [4]:
_TAG_TYPES = ["IP", "FS", "RP", "RM", "TH", "CF", "UC", "PL", "SP"]
VALID_LABELS = {"O", *(f"B-{t}" for t in _TAG_TYPES), *(f"I-{t}" for t in _TAG_TYPES)}


def validate_bio(toks: list[str], labels: list[str], record_id) -> list[str]:
    fixed = list(labels)
    prev_type = None
    for i, lb in enumerate(fixed):
        if lb not in VALID_LABELS:
            print(f"[{record_id}] unknown label {lb!r} at token {toks[i]!r}, coercing to O")
            fixed[i] = "O"
            lb = "O"
        if lb.startswith("I-"):
            tag_type = lb[2:]
            if prev_type != tag_type:
                print(f"[{record_id}] orphan {lb} at token {toks[i]!r}, coercing to B-{tag_type}")
                fixed[i] = f"B-{tag_type}"
        prev_type = fixed[i][2:] if fixed[i] != "O" else None
    return fixed

## Step 5 — Run labeler over the corpus

In [5]:
rows = [json.loads(line) for line in SRC.open()]

labeled_rows = []
tag_counts = Counter()
unmatched = []
for row in rows:
    toks, labels = full_label(row["text_normalized"], row.get("disfluencies", []))
    if labels is None:
        unmatched.append(row["id"])
        continue
    labels = validate_bio(toks, labels, row["id"])
    for lb in labels:
        tag_counts[lb] += 1
    labeled_rows.append({
        "id": row["id"],
        "text": row["text_normalized"],
        "intents": row["intents"],
        "tokens": toks,
        "labels": labels,
    })

print(f"{len(labeled_rows)} records labeled, {len(unmatched)} unmatched: {unmatched}")
print("tag counts:", dict(tag_counts))

1931 records labeled, 12 unmatched: [131, '131_aug1', '131_aug2', 157, '157_aug1', '157_aug2', 210, '210_aug1', '210_aug2', 551, '551_aug1', '551_aug2']
tag counts: {'B-IP': 673, 'I-IP': 687, 'O': 41358, 'B-SP': 375, 'B-FS': 293, 'I-FS': 500, 'B-RM': 1256, 'B-TH': 362, 'I-TH': 639, 'I-RM': 1576, 'B-PL': 332, 'B-RP': 304, 'B-UC': 285, 'B-CF': 318, 'I-CF': 342, 'I-SP': 66, 'I-UC': 192, 'I-PL': 95}


## Step 6 — Sanity check on sample records

Spot-checks the gold-to-BIO mapping against representative records for each tag, including a multi-word `eh ya` `IP` span and a genuine `RP`/`RM` repair pair.

In [6]:
sample_ids = [1, 3, 267, 7]
samples_by_id = {row["id"]: row for row in rows}

for rid in sample_ids:
    row = samples_by_id[rid]
    toks, labels = full_label(row["text_normalized"], row.get("disfluencies", []))
    print(row["text_normalized"])
    print([f"{t}/{l}" for t, l in zip(toks, labels) if l != "O"] or "no tags")
    print()

eh saya mau eh yaudah tambah nasi campur satu lagi eh saya mau juga ayam bakar satu eh yaudah deh
['eh/B-IP', 'saya/I-IP', 'mau/I-IP', 'yaudah/B-SP']

eh nanti ganti deh bukan bebek bakar th saya diskusi dulu mau lontong sayur aja mungkin tambah porsi ya
['saya/B-TH', 'diskusi/I-TH', 'dulu/I-TH']

eh mohon tahu dong ya soto dagingnya berapa harga sih saya mikir dulu nih mungkin tambah es sedikit juga ya
['mohon/B-PL', 'dong/B-SP', 'ya/I-SP', 'saya/B-IP', 'mikir/I-IP', 'dulu/I-IP']

eh nanti ya gak mau pesen mie ayamtambah jamur lagi eh berubah jadi kentang goreng aja ya mie mie kan entah nanti mau tambah apa lagi
['mie/B-RP', 'eh/B-RM', 'berubah/I-RM', 'mie/B-RM', 'entah/B-UC']



## Step 7 — Align labels to IndoBERT WordPiece tokens

`indobenchmark/indobert-base-p1` — plain IndoBERT, matches the NER backbone (reverted in `9f1dd1a`). Word-level BIO labels align to WordPiece subwords via `word_ids()`: the first subword of each word keeps its label id, continuation subwords and special tokens get `-100` so `CrossEntropyLoss` ignores them during training.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_list = sorted({lb for r in labeled_rows for lb in r["labels"]})
label2id = {lb: i for i, lb in enumerate(label_list)}
id2label = {i: lb for lb, i in label2id.items()}
print(label2id)

{'B-CF': 0, 'B-FS': 1, 'B-IP': 2, 'B-PL': 3, 'B-RM': 4, 'B-RP': 5, 'B-SP': 6, 'B-TH': 7, 'B-UC': 8, 'I-CF': 9, 'I-FS': 10, 'I-IP': 11, 'I-PL': 12, 'I-RM': 13, 'I-SP': 14, 'I-TH': 15, 'I-UC': 16, 'O': 17}


In [8]:
def align_labels_with_tokens(tokens: list[str], labels: list[str]) -> dict:
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    word_ids = encoding.word_ids()
    aligned = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif word_id != prev_word_id:
            aligned.append(label2id[labels[word_id]])
        else:
            aligned.append(-100)
        prev_word_id = word_id
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": aligned,
    }


aligned_rows = []
mismatches = 0
for r in labeled_rows:
    aligned = align_labels_with_tokens(r["tokens"], r["labels"])
    if len(aligned["input_ids"]) != len(aligned["labels"]):
        mismatches += 1
        continue
    aligned_rows.append({
        "id": r["id"],
        "intents": r["intents"],
        **aligned,
    })

print(f"{len(aligned_rows)} aligned, {mismatches} mismatches")

1931 aligned, 0 mismatches


## Step 8 — Dominant disfluency type per record (for stratified split)

Each record gets one dominant tag for stratification — the rarest non-`O` tag type present, falling back to `fluent` for records with no disfluency tags. Prioritizing the rarest tag (rather than e.g. first-occurring) keeps low-count categories from being randomly orphaned into a single split. `TAG_PRIORITY` order is a prior guess at rarity (structural/correction tags first, common fillers/particles last) — re-check against the printed `Counter` below after regenerating data and reorder if a tag assumed rare turns out common (or vice versa).

In [9]:
TAG_PRIORITY = ["FS", "RP", "RM", "UC", "CF", "TH", "PL", "SP", "IP"]


def dominant_type(labels: list[str]) -> str:
    present = {lb[2:] for lb in labels if lb != "O"}
    for tag in TAG_PRIORITY:
        if tag in present:
            return tag
    return "fluent"


dominant_by_id = {r["id"]: dominant_type(r["labels"]) for r in labeled_rows}
print(Counter(dominant_by_id.values()))

Counter({'RM': 624, 'FS': 293, 'RP': 263, 'CF': 166, 'UC': 136, 'TH': 130, 'IP': 120, 'SP': 106, 'PL': 86, 'fluent': 7})


## Step 9 — Stratified 80/10/10 split

Some tag types may only have a handful of records. `train_test_split` needs >=2 members per class per split, and this runs as *two* sequential splits (80/20, then the 20% into 10/10) — a class needs >=4 total to survive both. Records whose dominant type has fewer than `MIN_STRATIFY_COUNT` occurrences go entirely to train; the rest stratify normally.

In [10]:
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRAC = 0.15
TEST_FRAC = 0.15
MIN_STRATIFY_COUNT = 4

counts = Counter(dominant_by_id.values())
forced_train_ids = {rid for rid, dt in dominant_by_id.items() if counts[dt] < MIN_STRATIFY_COUNT}
stratifiable = [r for r in aligned_rows if r["id"] not in forced_train_ids]
forced_train = [r for r in aligned_rows if r["id"] in forced_train_ids]

labels_strat = [dominant_by_id[r["id"]] for r in stratifiable]
train, rest = train_test_split(
    stratifiable, test_size=VAL_FRAC + TEST_FRAC, stratify=labels_strat, random_state=SEED
)
rest_labels = [dominant_by_id[r["id"]] for r in rest]
val, test = train_test_split(
    rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=rest_labels, random_state=SEED
)
train = train + forced_train

print(f"train={len(train)} val={len(val)} test={len(test)}")

train=1351 val=290 test=290


In [11]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


OUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
json.dump(label2id, (OUT_DIR / "label_map.json").open("w"), indent=2)

print("wrote", OUT_DIR)

wrote ../data/disfluency
